# 03 · Evaluation

Computes mAP, per-class precision/recall/F1, confusion matrix, PR curves, inference speed, a prediction grid, failure cases and training curves for the trained model on the **test** split.

> **Runtime:** go to *Runtime → Change runtime type → T4 GPU* before running anything.
> Every cell below is safe to re-run; nothing is lost when Colab disconnects because all
> data, checkpoints and results live on your Google Drive.

## 0.1 GPU check
**What:** prints the GPU Colab assigned to this session.
**Why:** training on CPU takes hours instead of minutes; we warn loudly if no GPU is present.

In [ ]:
import subprocess, shutil

try:
    if shutil.which("nvidia-smi") is None:
        raise FileNotFoundError("nvidia-smi not found")
    print(subprocess.check_output(["nvidia-smi"], encoding="utf-8", errors="replace"))
    GPU_AVAILABLE = True
except Exception as exc:
    GPU_AVAILABLE = False
    print("=" * 70)
    print("WARNING: No GPU detected (", exc, ")")
    print("Go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU, then re-run.")
    print("=" * 70)

## 0.2 Mount Google Drive & get the code
**What:** mounts Drive at `/content/drive`, then either uses a copy of the repo already on Drive
or clones it from GitHub into `/content`.
**Why:** Drive is the only storage that survives a runtime disconnect. Checkpoints, the prepared
dataset and evaluation outputs are all written under `SAVE_DIR`.

Edit `REPO_URL` once (your fork), or copy the repo folder to
`MyDrive/masked-face-detection/repo` and it will be picked up automatically.

In [ ]:
import os, sys

REPO_URL = "https://github.com/Numbu-bit/masked-face-detection.git"   # <-- change if you fork
SAVE_DIR = "/content/drive/MyDrive/masked-face-detection"                  # everything persistent lives here
DRIVE_REPO = os.path.join(SAVE_DIR, "repo")
LOCAL_REPO = "/content/masked-face-detection"

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not in Colab - using the current working directory as the repo.")
    SAVE_DIR = os.path.abspath("runs")

os.makedirs(SAVE_DIR, exist_ok=True)

if IN_COLAB:
    if os.path.isfile(os.path.join(DRIVE_REPO, "configs", "default.yaml")):
        REPO_DIR = DRIVE_REPO
        print("Using repo copy on Drive:", REPO_DIR)
    else:
        REPO_DIR = LOCAL_REPO
        if not os.path.isfile(os.path.join(REPO_DIR, "configs", "default.yaml")):
            if "YOUR_USERNAME" in REPO_URL:
                raise RuntimeError(
                    "Set REPO_URL to your GitHub fork above, OR copy the repository folder to "
                    f"{DRIVE_REPO} so the notebook can find configs/default.yaml.")
            rc = os.system(f"git clone -q {REPO_URL} {REPO_DIR}")
            if rc != 0:
                raise RuntimeError(f"git clone failed for {REPO_URL}. Is the repo public / URL correct?")
        else:
            os.system(f"git -C {REPO_DIR} pull -q")
else:
    REPO_DIR = os.getcwd() if os.path.isfile("configs/default.yaml") else os.path.abspath("..")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("Repo:", REPO_DIR)
print("Persistent storage:", SAVE_DIR)

## 0.3 Install dependencies
**What:** installs the tested stack from `requirements.txt`. If that cannot be installed on this
runtime's Python version, it automatically falls back to `requirements-fallback.txt`
(same libraries, range pins) and tells you so.
**Why:** exact pins avoid the "it worked yesterday" class of breakages, but Colab upgrades its
Python from time to time and old pins may have no wheels for it — the fallback keeps you running.
Takes ~1–2 minutes on a fresh runtime; instant on re-runs.

In [ ]:
import subprocess, sys, platform

print("Python", platform.python_version())


def pip_install(req_file: str) -> "subprocess.CompletedProcess":
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file],
                          capture_output=True, encoding="utf-8", errors="replace")


proc = pip_install("requirements.txt")
if proc.returncode == 0:
    print("Dependencies installed (requirements.txt).")
else:
    print("requirements.txt could not be installed on this runtime. pip said:\n")
    print(proc.stderr[-2500:])
    print("\n-> Trying requirements-fallback.txt (range pins) ...")
    proc = pip_install("requirements-fallback.txt")
    if proc.returncode == 0:
        print("Dependencies installed (requirements-fallback.txt).")
    else:
        print(proc.stderr[-2500:])
        raise RuntimeError("Both requirement sets failed. Copy the pip output above into an issue / to Claude.")

import ultralytics, torch, numpy, cv2
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| numpy", numpy.__version__,
      "| opencv", cv2.__version__, "| CUDA", torch.cuda.is_available())

## 0.4 Seeds & config
**What:** loads `configs/default.yaml` (the single source of truth for every hyper-parameter)
and seeds Python / NumPy / PyTorch / CUDA with `seed=42`.
**Why:** reproducible splits, reproducible training.

In [ ]:
import json
from src.utils import load_config, set_seed, get_device, resolve_save_dir

# Optional: MFD_OVERRIDES='{"epochs": 1}' in the environment (used by automated tests)
ENV_OVERRIDES = json.loads(os.environ.get("MFD_OVERRIDES", "{}"))
cfg = load_config(overrides={"save_dir": os.path.join(SAVE_DIR, "runs"), **ENV_OVERRIDES})
set_seed(cfg["seed"])
device = get_device()
RUN_DIR = resolve_save_dir(cfg) / cfg["run_name"]
DATA_ROOT = cfg["data_root"]
print("Model:", cfg["model_variant"], "| image size:", cfg["image_size"], "| batch:", cfg["batch_size"])
print("Run directory:", RUN_DIR)

## Restore the prepared dataset
**What:** if `/content/data` is missing (new runtime), unzips the copy that notebook 01 saved to Drive.
**Why:** `/content` is wiped on every disconnect; Drive is not.

In [ ]:
import zipfile
from pathlib import Path

DATA_ZIP = Path(SAVE_DIR) / "data_prepared.zip"
DATA_YAML = Path(DATA_ROOT) / "data.yaml"

if not DATA_YAML.exists():
    if DATA_ZIP.exists():
        print("Restoring dataset from", DATA_ZIP, "...")
        with zipfile.ZipFile(DATA_ZIP) as zf:
            zf.extractall(Path(DATA_ROOT).parent)
        print("Restored.")
    else:
        raise FileNotFoundError(
            f"Neither {DATA_YAML} nor {DATA_ZIP} exists. Run notebook 01_data_preparation.ipynb first.")

from src.dataset import dataset_statistics
import pandas as pd
display(pd.DataFrame(dataset_statistics(Path(DATA_ROOT), cfg["class_names"])).T)

## 1. Load the trained model
**What:** loads `best.pt` from the run directory (edit `WEIGHTS` to evaluate a different checkpoint).

In [ ]:
from pathlib import Path
from src.model import load_trained_yolo, model_summary

WEIGHTS = None   # e.g. "/content/drive/MyDrive/masked-face-detection/runs/yolov8s_mask/weights/best.pt"
model = load_trained_yolo(cfg, WEIGHTS)
print(model_summary(model))

## 2. Metrics: mAP@0.5, mAP@0.5:0.95, per-class P / R / F1
**What:** runs Ultralytics validation on the test split (conf 0.001, IoU 0.5) and tabulates results.
**Why:** mAP is the headline detection metric; per-class F1 shows whether the rare
`mask_worn_incorrectly` class actually works.

In [ ]:
from src.evaluate import evaluate_yolo, print_report
import pandas as pd

report = evaluate_yolo(model, cfg, str(DATA_YAML), split="test")
print_report(report, cfg)
per_class = pd.DataFrame(report.per_class).T
display(per_class.style.format("{:.3f}"))

## 3. Confusion matrix (3 classes + background)
**What:** normalised heat-map; rows = predicted, columns = true.
**Why:** the *background* row/column reveals missed faces vs. false alarms, and the off-diagonal
cells show which mask states get confused with each other.

In [ ]:
import matplotlib.pyplot as plt
from src.evaluate import plot_confusion_matrix
plot_confusion_matrix(report, cfg, normalize=True); plt.show()

## 4. Precision–Recall curves per class

In [ ]:
from src.evaluate import plot_pr_curves
plot_pr_curves(report, cfg); plt.show()

## 5. Inference speed (FPS) on GPU and CPU
**What:** times 50 end-to-end predictions (pre-process + forward + NMS) per device.
**Why:** the project target is ≥ 30 FPS on a T4.

In [ ]:
from src.evaluate import benchmark_fps
fps = benchmark_fps(model, cfg, Path(DATA_ROOT), n_images=50, warmup=5)
print({k: round(v, 1) for k, v in fps.items()})

## 6. Prediction grid — 16 random test images
Colour code: **green** = with_mask, **red** = without_mask, **orange** = mask_worn_incorrectly.

In [ ]:
from src.evaluate import plot_prediction_grid
plot_prediction_grid(model, cfg, Path(DATA_ROOT), split="test", n=16, seed=cfg["seed"]); plt.show()

## 7. Failure-case analysis
**What:** matches every prediction (conf ≥ 0.25) to ground truth (IoU ≥ 0.5) and shows the
10 *lowest-confidence correct* and 10 *highest-confidence wrong* detections.
**Why:** the wrong-but-confident row is where labelling errors and genuinely hard cases live —
this is what to look at before deciding to collect more data.

In [ ]:
from src.evaluate import collect_detection_cases, plot_failure_cases
cases = collect_detection_cases(model, cfg, Path(DATA_ROOT), split="test", conf=0.25)
n_ok = sum(c.correct for c in cases)
print(f"{len(cases)} detections analysed: {n_ok} correct, {len(cases) - n_ok} wrong")
plot_failure_cases(cases, cfg, k=10); plt.show()

## 8. Training curves (loss, mAP, learning rate)
Pulled from Ultralytics' `results.csv` in the run directory.

In [ ]:
from src.evaluate import plot_training_curves
plot_training_curves(cfg); plt.show()

## 9. Did we hit the targets?
Targets live in `configs/default.yaml → targets` (mAP@0.5 ≥ 0.85, per-class F1 ≥ 0.80, ≥ 30 FPS on GPU).
If anything falls short, concrete next steps are printed. A `metrics_test.json` and all figures
were saved to `RUN_DIR/eval/` on Drive — paste the numbers into the README table.

In [ ]:
from src.evaluate import check_targets
import json

tips = check_targets(report, fps, cfg)
if tips:
    print("Targets NOT fully met. Suggestions:")
    for t in tips:
        print(" -", t)
else:
    print("All targets met.")

summary = {"model": cfg["model_variant"], "image_size": cfg["image_size"],
           "mAP50": round(report.map50, 4), "mAP50_95": round(report.map50_95, 4),
           "per_class_f1": {k: round(v["f1"], 3) for k, v in report.per_class.items()},
           "gpu_fps": round(fps["gpu_fps"], 1), "cpu_fps": round(fps["cpu_fps"], 1)}
print(json.dumps(summary, indent=2))
print("Figures saved under:", report.save_dir)